In [ ]:
!pip install torch_geometric

In [ ]:
import pandas as pd
import torch
import os
import numpy as np
from torch_geometric.data import Data


In [ ]:
# Change directory
os.chdir('/content/drive/MyDrive/nids-mitre/')

!pwd


/content/drive/MyDrive/nids-mitre


In [ ]:
# --- CONFIGURATION ---
CSV_PATH = "/content/drive/MyDrive/nids-mitre/data/cicids2018-v3/cicids2018v3_wed2802.csv"
BASE_OUTPUT_DIR = "./dataset_processed"
TIME_WINDOW_SEC = 30  # Each graph represents 30 seconds
CHUNK_SIZE = 1000000   # Rows to read at a time in RAM


In [ ]:
cicids2018v3_wed2802_orig = pd.read_csv(CSV_PATH)
cicids2018v3_wed2802 = cicids2018v3_wed2802_orig.copy()
# Convert 'FLOW_START_TIME' to datetime type after loading
cicids2018v3_wed2802['FLOW_START_TIME'] = pd.to_datetime(cicids2018v3_wed2802['FLOW_START_TIME'])

timestamps = cicids2018v3_wed2802['FLOW_START_TIME']
GLOBAL_START_TIME = timestamps.min()
GLOBAL_END_TIME = timestamps.max()
TOTAL_DURATION = GLOBAL_END_TIME - GLOBAL_START_TIME

# Limits for Train/Val/Test
LIMIT_TRAIN = GLOBAL_START_TIME + (TOTAL_DURATION * 0.70)
LIMIT_VAL = GLOBAL_START_TIME + (TOTAL_DURATION * 0.85)

print(f"Global start time: {GLOBAL_START_TIME}")
print(f"Global end time: {GLOBAL_END_TIME}")

print(f"Limit train: {LIMIT_TRAIN}")
print(f"Limit val: {LIMIT_VAL}")


In [ ]:
# SELECTED FEATURES
SELECTED_FEATURES = [
    'IN_BYTES', 'OUT_BYTES', 'IN_PKTS', 'OUT_PKTS',
    'FLOW_DURATION_MILLISECONDS', 'DURATION_IN', 'DURATION_OUT',
    'SRC_TO_DST_IAT_AVG', 'DST_TO_SRC_IAT_AVG',
    'SRC_TO_DST_IAT_STDDEV', 'DST_TO_SRC_IAT_STDDEV',
    'MIN_IP_PKT_LEN', 'MAX_IP_PKT_LEN',
    'RETRANSMITTED_IN_PKTS', 'RETRANSMITTED_OUT_PKTS',
    'TCP_WIN_MAX_IN', 'TCP_WIN_MAX_OUT',
    'TCP_FLAGS', 'MIN_TTL', 'MAX_TTL'
]

In [ ]:
# Preparar directorios
splits = ['train', 'val', 'test']
for s in splits:
    os.makedirs(os.path.join(BASE_OUTPUT_DIR, s), exist_ok=True)


In [ ]:
import gc

del cicids2018v3_wed2802_orig
del cicids2018v3_wed2802
gc.collect()

442

In [ ]:
# --- 2. GESTIÓN DE IPs (Inductiva) ---
ip_map = {}
next_id = 0

def get_ip_id(ip_str):
    global next_id
    if ip_str not in ip_map:
        ip_map[ip_str] = next_id
        next_id += 1
    return ip_map[ip_str]


# Función auxiliar para guardar grafos VACÍOS
def save_empty_graph(window_id):
    # Determinar split (igual que antes)
    window_start_time = GLOBAL_START_TIME + pd.Timedelta(seconds=window_id * TIME_WINDOW_SEC)
    if window_start_time < LIMIT_TRAIN: split = 'train'
    elif window_start_time < LIMIT_VAL: split = 'val'
    else: split = 'test'

    # Crear Grafo Vacío
    # Tensores vacíos con la forma correcta (0 filas, N columnas)
    x = torch.empty((0, 16), dtype=torch.float)
    edge_index = torch.empty((2, 0), dtype=torch.long)
    edge_attr = torch.empty((0, len(SELECTED_FEATURES)), dtype=torch.float)
    y = torch.empty((0), dtype=torch.float) # Sin etiquetas

    data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y)
    data.global_node_ids = torch.empty((0), dtype=torch.long)
    data.timestamp = window_start_time
    data.is_empty = True # Flag útil para el modelo

    filename = f"graph_{int(window_id):06d}.pt"
    torch.save(data, os.path.join(BASE_OUTPUT_DIR, split, filename))


def save_window_group(window_id, df_group):
    """Procesa un grupo de Pandas que ya pertenece a una única ventana temporal"""

    # Determinar a qué split pertenece basándonos en el tiempo de INICIO de la ventana
    window_start_time = GLOBAL_START_TIME + pd.Timedelta(seconds=window_id * TIME_WINDOW_SEC)

    if window_start_time < LIMIT_TRAIN:
        split = 'train'
    elif window_start_time < LIMIT_VAL:
        split = 'val'
    else:
        split = 'test'

    # --- Construcción del Grafo ---
    src_ids = [get_ip_id(ip) for ip in df_group['IPV4_SRC_ADDR']]
    dst_ids = [get_ip_id(ip) for ip in df_group['IPV4_DST_ADDR']]

    edge_index = torch.tensor([src_ids, dst_ids], dtype=torch.long)

    # Features y Normalización
    feats = torch.tensor(df_group[SELECTED_FEATURES].values.astype('float32'))
    edge_attr = torch.log1p(feats)

    # Target
    y = torch.tensor(
        df_group['Attack'].apply(lambda x: 1 if x == 'Infilteration' else 0).values,
        dtype=torch.float
    )

    # Nodes (Identity Agnostic)
    unique_nodes = torch.tensor(list(set(src_ids + dst_ids)), dtype=torch.long)
    num_nodes = len(unique_nodes)
    x = torch.ones((num_nodes, 16), dtype=torch.float)

    data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y)
    data.global_node_ids = unique_nodes
    data.timestamp = window_start_time # Guardamos metadata útil

    # Guardar: Usamos el window_id como nombre para mantener orden estricto
    # graph_00000.pt, graph_00001.pt ...
    filename = f"graph_{int(window_id):06d}.pt"
    torch.save(data, os.path.join(BASE_OUTPUT_DIR, split, filename))

In [ ]:
# --- 3. BUCLE DE PROCESAMIENTO OPTIMIZADO ---
expected_window_id = 0 # Contador para rastrear continuidad
buffer_df = pd.DataFrame()
print("Procesando chunks...")

# Iterador de chunks
chunk_iterator = pd.read_csv(CSV_PATH, chunksize=CHUNK_SIZE)

for chunk_idx, chunk in enumerate(chunk_iterator):
    # 1. Unir con buffer anterior (si existe)
    df = pd.concat([buffer_df, chunk])
    df['FLOW_START_TIME'] = pd.to_datetime(df['FLOW_START_TIME'])

    # 2. Asignar ID de Ventana a cada fila (Vectorizado = Muy Rápido)
    # Restamos el inicio global y dividimos por 30s. El floor (int) nos da el ID.
    df['window_id'] = ((df['FLOW_START_TIME'] - GLOBAL_START_TIME) // pd.Timedelta(seconds=TIME_WINDOW_SEC)).astype(int)

    # 3. Identificar qué ventanas tenemos en este chunk
    present_windows = df['window_id'].unique()
    present_windows.sort()

    if len(present_windows) > 0:
        # 4. ESTRATEGIA DE SEGURIDAD:
        # La última ventana presente en este chunk podría estar incompleta.
        # (Sus datos podrían continuar en el siguiente chunk).
        # Por tanto, procesamos todas MENOS la última.
        last_window_id = present_windows[-1]

        # Separar: Completas vs Incompleta
        df_complete = df[df['window_id'] < last_window_id]
        df_incomplete = df[df['window_id'] == last_window_id]

        # 5. Procesar grupos completos
        # groupby es mucho más eficiente que filtrar una y otra vez
        for win_id, group in df_complete.groupby('window_id'):
            # 1. Rellenar huecos si saltamos números
            while expected_window_id < win_id:
                save_empty_graph(expected_window_id)
                expected_window_id += 1

            # 2. Guardar la ventana actual (que sí tiene datos)
            save_window_group(win_id, group)
            expected_window_id += 1

        # 6. Guardar lo incompleto en el buffer para la siguiente vuelta
        buffer_df = df_incomplete
    else:
        # Caso raro: Todo el chunk pertenece a una ventana que ya estaba en el buffer
        buffer_df = df

    print(f"Chunk {chunk_idx} procesado. Buffer size: {len(buffer_df)}")

Procesando chunks...
Chunk 0 procesado. Buffer size: 985
Chunk 1 procesado. Buffer size: 308
Chunk 2 procesado. Buffer size: 3


In [ ]:
# --- 4. LIMPIEZA FINAL ---
# Al terminar el CSV, lo que quede en el buffer es la última ventana válida.
# Asegurarse de que `expected_window_id` sea global y arrastre el estado de `Czdd5ayNQhju`
if not buffer_df.empty:
    print("Procesando último buffer...")
    for win_id, group in buffer_df.groupby('window_id'):
        # Rellenar huecos antes de la última ventana, si los hay
        while expected_window_id < win_id:
            save_empty_graph(expected_window_id)
            expected_window_id += 1
        save_window_group(win_id, group)
        expected_window_id += 1

# Rellenar cualquier hueco final después del último grafo procesado hasta el final del dataset
# Esto asume que GLOBAL_END_TIME se ha calculado correctamente al inicio.
# Calculemos el ID de la última ventana posible
final_possible_window_id = ((GLOBAL_END_TIME - GLOBAL_START_TIME) // pd.Timedelta(seconds=TIME_WINDOW_SEC))
while expected_window_id <= final_possible_window_id:
    save_empty_graph(expected_window_id)
    expected_window_id += 1

print("¡Generación de dataset completada!")

Procesando último buffer...
¡Generación de dataset completada!


## Verificaciones

In [ ]:
import torch
import os
import random

# Ajusta esto a tu ruta
ROOT_DIR = "./dataset_processed/train"

# Elegir un archivo al azar que NO esté vacío (para ver datos reales)
files = sorted([f for f in os.listdir(ROOT_DIR) if f.endswith('.pt')])

# Intentamos buscar uno no vacío para el ejemplo
sample_data = None
for f in files:
    data = torch.load(os.path.join(ROOT_DIR, f), weights_only=False)
    if data.x.shape[0] > 0: # Si tiene nodos
        sample_data = data
        print(f"Analizando archivo: {f}")
        break

if sample_data:
    print("="*40)
    print(f"Keys disponibles: {sample_data.keys}")
    print(f"Nodos (x): {sample_data.x.shape} -> (Num Nodos, 16 features dummy)")
    print(f"Aristas (edge_index): {sample_data.edge_index.shape} -> (2, Num Flows)")
    print(f"Features Arista (edge_attr): {sample_data.edge_attr.shape} -> (Num Flows, {sample_data.edge_attr.shape[1]})")
    print(f"Labels (y): {sample_data.y.shape} -> (Num Flows,)")
    print(f"Timestamp: {sample_data.timestamp}")
    print(f"Es vacío?: {getattr(sample_data, 'is_empty', False)}")

    # Check de coherencia
    num_edges = sample_data.edge_index.shape[1]
    num_labels = sample_data.y.shape[0]
    num_attrs = sample_data.edge_attr.shape[0]

    if num_edges == num_labels == num_attrs:
        print("✅ COHERENCIA: Nro de aristas coincide con features y labels.")
    else:
        print("❌ ERROR: Desajuste en dimensiones de aristas.")

    # Ver si hay algún ataque en este snapshot
    attacks = sample_data.y.sum().item()
    print(f"Cantidad de flujos maliciosos en este grafo: {int(attacks)}")
else:
    print("Solo encontré grafos vacíos o no hay archivos.")

Analizando archivo: graph_000000.pt
Keys disponibles: <bound method BaseData.keys of Data(x=[9, 16], edge_index=[2, 8], edge_attr=[8, 22], y=[8], global_node_ids=[9], timestamp=2018-02-28 00:12:55.854000)>
Nodos (x): torch.Size([9, 16]) -> (Num Nodos, 16 features dummy)
Aristas (edge_index): torch.Size([2, 8]) -> (2, Num Flows)
Features Arista (edge_attr): torch.Size([8, 22]) -> (Num Flows, 22)
Labels (y): torch.Size([8]) -> (Num Flows,)
Timestamp: 2018-02-28 00:12:55.854000
Es vacío?: False
✅ COHERENCIA: Nro de aristas coincide con features y labels.
Cantidad de flujos maliciosos en este grafo: 0


In [ ]:
import pandas as pd

def check_temporal_continuity(split_name='train'):
    dir_path = os.path.join("./dataset_processed", split_name)
    files = sorted([f for f in os.listdir(dir_path) if f.endswith('.pt')])

    print(f"\nVerificando continuidad en split: {split_name} ({len(files)} archivos)...")

    prev_id = -1
    prev_time = None

    errors = 0
    empty_graphs = 0

    # Revisamos solo los primeros 1000 para no tardar mucho, o quita el [:1000] para todo
    for f in files[:1000]:
        # Extraer ID del nombre "graph_000123.pt"
        curr_id = int(f.split('_')[1].split('.')[0])

        data = torch.load(os.path.join(dir_path, f), weights_only=False)
        curr_time = data.timestamp

        # 1. Verificar ID consecutivo
        if prev_id != -1:
            if curr_id != prev_id + 1:
                print(f"❌ SALTO DETECTADO: De {prev_id} pasamos a {curr_id}. Faltan archivos.")
                errors += 1

        # 2. Verificar Tiempo (debe ser exactamente +30s)
        # Nota: curr_time puede ser Timestamp o string, aseguramos conversion
        if prev_time is not None:
            diff = (curr_time - prev_time).total_seconds()
            if diff != 30.0:
                 print(f"⚠️ ALERTA TIEMPO: Grafo {curr_id} salto de {diff}s (esperado 30s)")

        # 3. Contar vacíos
        if data.x.shape[0] == 0:
            empty_graphs += 1

        prev_id = curr_id
        prev_time = curr_time

    if errors == 0:
        print(f"✅ Secuencia numérica CORRECTA (IDs consecutivos).")
    print(f"ℹ️ Total grafos vacíos encontrados (relleno): {empty_graphs}")

check_temporal_continuity('train')
# check_temporal_continuity('test')


Verificando continuidad en split: train (1998 archivos)...
✅ Secuencia numérica CORRECTA (IDs consecutivos).
ℹ️ Total grafos vacíos encontrados (relleno): 824


In [ ]:
def check_content_stats(split_name='train'):
    dir_path = os.path.join("./dataset_processed", split_name)
    files = sorted([f for f in os.listdir(dir_path) if f.endswith('.pt')])

    total_flows = 0
    total_attacks = 0
    has_nans = False

    # Muestreo aleatorio de 50 grafos para no saturar
    sample_files = random.sample(files, min(len(files), 50))

    print(f"\nAnalizando contenido (Muestra de {len(sample_files)} grafos)...")

    for f in sample_files:
        data = torch.load(os.path.join(dir_path, f), weights_only=False)

        if data.x.shape[0] == 0: continue # Saltar vacíos

        # Chequear NaNs
        if torch.isnan(data.edge_attr).any() or torch.isnan(data.x).any():
            print(f"❌ NaN detectado en {f}")
            has_nans = True

        # Chequear Infs
        if torch.isinf(data.edge_attr).any():
             print(f"❌ Infinito detectado en {f} (Revisar log1p)")

        total_flows += data.y.shape[0]
        total_attacks += data.y.sum().item()

    print(f"Total Flujos Analizados: {total_flows}")
    print(f"Total Ataques Encontrados: {int(total_attacks)}")
    print(f"Porcentaje de Ataques: {(total_attacks/total_flows)*100:.4f}%")

    if not has_nans:
        print("✅ No se detectaron NaNs en la muestra.")

check_content_stats('train')


Analizando contenido (Muestra de 50 grafos)...
Total Flujos Analizados: 26896
Total Ataques Encontrados: 1167
Porcentaje de Ataques: 4.3389%
✅ No se detectaron NaNs en la muestra.
